# Expansion rate and the shape of the phase C response

Two questions, both from the released archive:

1. How fast does the repeat grow in each cell type, measured from the distribution of per-nucleus repeat lengths rather than assumed?
2. Past 150 CAGs, do the phase C genes move together as one program, or does each move on its own?

Runtime → Run all. The download takes 10–20 minutes. Everything written out is small.

In [ ]:
options(timeout = 7200)
for (p in c("data.table", "Matrix")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
suppressPackageStartupMessages({library(data.table); library(Matrix)})
sessionInfo()$R.version$version.string

## 1. Download the archive

In [ ]:
zip <- "HD_cortex.zip"
url <- "https://zenodo.org/records/18202664/files/HD_cortex.zip?download=1"
if (!file.exists(zip) || file.size(zip) < 2.5e9) {
  status <- system(paste("curl -L --fail -o", zip, shQuote(url)))
  if (status != 0) stop("Download failed. Upload HD_cortex.zip to the Files panel and rerun this cell.")
}
file.size(zip)

## 2. What is in the archive

Lists every file, then extracts the small ones. We are looking for donor-level information: age, and the repeat length each donor inherited.

In [ ]:
all_files <- system(paste("unzip -Z1", zip), intern = TRUE)
cat(length(all_files), "files\n\n")
writeLines(head(all_files, 200))
cat("\n--- files whose names suggest donor or metadata ---\n")
writeLines(grep("donor|meta|sample|table|clinical|S1|csv$|tsv$|txt$", all_files, value = TRUE, ignore.case = TRUE))

In [ ]:
small <- grep("csv$|tsv$|txt$|json$", all_files, value = TRUE)
if (length(small)) system(paste("unzip -o", zip, paste(shQuote(small), collapse = " ")))
for (f in small) {
  cat("\n=====", f, "\n")
  x <- tryCatch(fread(f, nrows = 3), error = function(e) NULL)
  if (!is.null(x)) print(names(x)) else writeLines(head(readLines(f, warn = FALSE), 3))
}

## 3. Per-nucleus repeat lengths

Every nucleus with a measured repeat, with its donor, region and cell type. This is the table the expansion rate is fitted to, so it must be the unselected set, not the subset enriched for long repeats.

In [ ]:
dirs <- grep("summary.rds$", all_files, value = TRUE)
cat("summary tables in the archive:\n"); writeLines(dirs)
system(paste("unzip -o", zip, paste(shQuote(dirs), collapse = " ")))

out <- rbindlist(lapply(dirs, function(f) {
  sd <- as.data.table(readRDS(f))
  cat("\n", f, ":", nrow(sd), "rows\n"); print(names(sd))
  keep <- intersect(c("CELL_BARCODE","DONOR","REGION","CAG_LENGTH","predClass","subcluster_name",
                      "predClass_allen_gluta_subclass","predClass_SPN_DI","NUM_TRANSCRIPTS","NUM_GENES"), names(sd))
  x <- sd[, ..keep]
  x[, SOURCE := f]
  x
}), fill = TRUE)
fwrite(out, "per_nucleus_repeat_lengths.csv.gz")
cat("\nwrote", nrow(out), "nuclei\n")
print(out[!is.na(CAG_LENGTH), .(n = .N, median = median(CAG_LENGTH), q95 = quantile(CAG_LENGTH, .95),
                                max = max(CAG_LENGTH)), by = SOURCE])

## 4. Do the phase C genes move together?

For every nucleus, each gene's observed count is divided by what that nucleus would be expected to show with no effect, taken from reference nuclei of 36 to 100 CAGs in the same donor and cell type. The question is whether those ratios rise and fall together across nuclei past 150 CAGs.

A set of control genes is matched to the phase C genes on how strongly they are expressed, because genes detected at similar levels carry similar noise. Anything the phase C genes do that the controls also do is not evidence of a program.

In [ ]:
dir1 <- "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic"
system(paste("unzip -o", zip, shQuote(file.path(dir1, "raw.dge.rds"))))
raw <- readRDS(file.path(dir1, "raw.dge.rds"))
sd  <- as.data.table(readRDS(file.path(dir1, "summary.rds")))
grp_col <- intersect(c("subcluster_name","predClass_allen_gluta_subclass"), names(sd))[1]
sd <- sd[!is.na(CAG_LENGTH) & CAG_LENGTH > 35 & CELL_BARCODE %in% colnames(raw)]
sd[, GROUP := paste(DONOR, get(grp_col), sep = "|")]

st <- fread("HD_cortex/data/gene_expr_models/cortex.stats.txt")
st[, SET := fifelse(!is.na(PHASE_C_PVALUE) & PHASE_C_PVALUE < 1e-10, fifelse(PHASE_C_EFFECT < 0, "C-", "C+"), "other")]
dgenes <- st[!is.na(PHASE_D_PVALUE) & PHASE_D_PVALUE < 1e-10, GENE]

tot  <- Matrix::rowSums(raw)
idx  <- match(sd$CELL_BARCODE, colnames(raw))
N    <- Matrix::colSums(raw)[idx]

cgenes <- intersect(st[SET != "other", GENE], rownames(raw))
# controls: for each phase C gene, the nearest unused gene by total expression
pool <- setdiff(rownames(raw)[tot > 0], union(cgenes, dgenes))
ord  <- pool[order(tot[pool])]
ctrl <- character(0)
for (g in cgenes) {
  j <- which.min(abs(tot[ord] - tot[g]))
  ctrl <- c(ctrl, ord[j]); ord <- ord[-j]
}
genes <- c(cgenes, ctrl)
rs <- raw[genes, , drop = FALSE]          # few rows, still sparse
X  <- as.matrix(rs[, idx, drop = FALSE])  # then the nuclei we need
rm(raw, rs); invisible(gc())
cat(length(cgenes), "phase C genes,", length(ctrl), "matched controls,", nrow(sd), "nuclei\n")

In [ ]:
is_ref <- sd$CAG_LENGTH <= 100
rate <- function(y) {
  r <- tapply(y[is_ref], sd$GROUP[is_ref], sum) / tapply(N[is_ref], sd$GROUP[is_ref], sum)
  v <- r[sd$GROUP]; v[is.na(v) | v <= 0] <- sum(y[is_ref]) / sum(N[is_ref]); as.numeric(v)
}
R <- t(apply(X, 1, function(y) y / pmax(N * rate(y), 1e-8)))   # observed over expected
rm(X); invisible(gc())

hi <- which(sd$CAG_LENGTH >= 150)
lo <- which(is_ref)
cat(length(hi), "nuclei at 150 CAGs or more;", length(lo), "reference nuclei\n")

summarise <- function(rows, cols, label) {
  M <- R[cols, rows, drop = FALSE]
  M <- M[apply(M, 1, function(v) sd(v) > 0), , drop = FALSE]
  C <- suppressWarnings(cor(t(M), method = "spearman"))
  off <- C[upper.tri(C)]
  pc <- prcomp(t(M), center = TRUE, scale. = TRUE)   # nuclei as observations, genes as variables
  ve <- pc$sdev^2 / sum(pc$sdev^2)
  data.table(set = label, genes = nrow(M), nuclei = length(rows),
             median_r = median(off, na.rm = TRUE), mean_r = mean(off, na.rm = TRUE),
             frac_r_above_0.2 = mean(off > 0.2, na.rm = TRUE),
             PC1 = ve[1], PC2 = ve[2], PC3 = ve[3])
}
res <- rbind(
  summarise(hi, cgenes, "phase C genes, nuclei >= 150"),
  summarise(hi, ctrl,   "matched controls, nuclei >= 150"),
  summarise(lo, cgenes, "phase C genes, reference nuclei"),
  summarise(lo, ctrl,   "matched controls, reference nuclei"))
print(res)
fwrite(res, "program_vs_independent.csv")

## 5. Per-nucleus scores

The combined phase C score for every nucleus, written out so its distribution can be examined here: one population drifting, or two populations with cells switching between them.

In [ ]:
score <- data.table(
  CAG_LENGTH = sd$CAG_LENGTH, DONOR = sd$DONOR, SUBTYPE = sd[[grp_col]], UMIs = N,
  Cplus  = colMeans(R[intersect(st[SET == "C+", GENE], rownames(R)), , drop = FALSE]),
  Cminus = colMeans(R[intersect(st[SET == "C-", GENE], rownames(R)), , drop = FALSE]),
  ctrl   = colMeans(R[ctrl, , drop = FALSE]))
fwrite(score, "per_nucleus_phaseC_scores.csv.gz")
print(score[, .(n = .N, Cplus = round(mean(Cplus), 3), Cminus = round(mean(Cminus), 3),
                ctrl = round(mean(ctrl), 3)),
            by = .(bin = cut(CAG_LENGTH, c(36,100,150,200,250,300,400,2000), right = FALSE))][order(bin)])

## 6. Download

Three files appear in the Files panel on the left: `per_nucleus_repeat_lengths.csv.gz`, `program_vs_independent.csv` and `per_nucleus_phaseC_scores.csv.gz`. Right-click each and choose Download.